# CSIT 598 - Assignment 1
## 06. Ensemble Methods (Bagging, Random Forest, AdaBoost, XGBoost)

Implementation of all four ensemble methods (Bagging, Random Forest, AdaBoost, XGBoost) and comparing their performance.

In [11]:
from pathlib import Path
import json
import numpy as np
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

from utils import evaluate_classifier, RANDOM_STATE

In [12]:
xgb_import_error = None
try:
    from xgboost import XGBClassifier
    xgb_available = True
except Exception as exc:
    xgb_available = False
    xgb_import_error = str(exc)

print(f'XGBoost available: {xgb_available}')
if xgb_import_error:
    print('XGBoost import error (will skip XGBoost run):')
    print(xgb_import_error)

XGBoost available: True


In [13]:
project_root = Path.cwd().parent
artifact_path = project_root / 'results' / 'metrics' / 'mnist_splits.npz'

if not artifact_path.exists():
    raise FileNotFoundError(
        f"Missing artifact: {artifact_path}. Run 01_data_preprocessing.ipynb first."
    )

data = dict(np.load(artifact_path))
X_train, y_train = data['X_train'], data['y_train']
X_val, y_val = data['X_val'], data['y_val']
X_test, y_test = data['X_test'], data['y_test']

X_trainval = np.vstack([X_train, X_val])
y_trainval = np.concatenate([y_train, y_val])

print('Loaded normalized data:')
print(f'X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}')

Loaded normalized data:
X_train: (50000, 784), X_val: (10000, 784), X_test: (10000, 784)


In [14]:
ensemble_results = {}

# 1) Bagging with Decision Trees
bagging_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=15, random_state=RANDOM_STATE),
    n_estimators=50,
    max_samples=0.7,
    n_jobs=-1,
    random_state=RANDOM_STATE,
 )
bagging_metrics = evaluate_classifier(bagging_model, X_trainval, y_trainval, X_test, y_test)
ensemble_results['bagging'] = bagging_metrics
print(f"Bagging accuracy: {bagging_metrics['accuracy']:.4f}")

# 2) Random Forest
rf_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=None,
    min_samples_split=2,
    n_jobs=-1,
    random_state=RANDOM_STATE,
 )
rf_metrics = evaluate_classifier(rf_model, X_trainval, y_trainval, X_test, y_test)
ensemble_results['random_forest'] = rf_metrics
print(f"Random Forest accuracy: {rf_metrics['accuracy']:.4f}")

# 3) AdaBoost with Decision Tree base estimator
ada_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=2, random_state=RANDOM_STATE),
    n_estimators=120,
    learning_rate=0.8,
    random_state=RANDOM_STATE,
 )
ada_metrics = evaluate_classifier(ada_model, X_trainval, y_trainval, X_test, y_test)
ensemble_results['adaboost'] = ada_metrics
print(f"AdaBoost accuracy: {ada_metrics['accuracy']:.4f}")

# 4) XGBoost with Decision Trees (if available)
if xgb_available:
    X_xgb_train, _, y_xgb_train, _ = train_test_split(
        X_trainval, y_trainval,
        train_size=30000,
        stratify=y_trainval,
        random_state=RANDOM_STATE,
    )
    xgb_model = XGBClassifier(
        objective='multi:softmax',
        num_class=10,
        n_estimators=150,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method='hist',
        eval_metric='mlogloss',
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
    xgb_metrics = evaluate_classifier(xgb_model, X_xgb_train, y_xgb_train, X_test, y_test)
    xgb_metrics['train_subset_size'] = int(len(y_xgb_train))
    ensemble_results['xgboost'] = xgb_metrics
    print(f"XGBoost accuracy: {xgb_metrics['accuracy']:.4f}")
else:
    ensemble_results['xgboost'] = None
    print('XGBoost unavailable; skipping XGBoost run.')

Bagging accuracy: 0.9487
Random Forest accuracy: 0.9687
AdaBoost accuracy: 0.8324
XGBoost accuracy: 0.9695


In [15]:
metrics_dir = project_root / 'results' / 'metrics'
metrics_dir.mkdir(parents=True, exist_ok=True)

def save_metrics(name, metrics, extra=None):
    if metrics is None:
        return
    payload = {k: v for k, v in metrics.items() if k != 'y_pred'}
    payload['model_name'] = name
    if extra:
        payload.update(extra)
    with open(metrics_dir / f'{name}_metrics.json', 'w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2)

save_metrics('bagging', ensemble_results['bagging'], {
    'base_estimator': 'DecisionTree(max_depth=15)',
    'n_estimators': 50,
    'max_samples': 0.7,
})

save_metrics('random_forest', ensemble_results['random_forest'], {
    'n_estimators': 150,
    'max_depth': None,
})

save_metrics('adaboost', ensemble_results['adaboost'], {
    'base_estimator': 'DecisionTree(max_depth=2)',
    'n_estimators': 120,
    'learning_rate': 0.8,
})

if ensemble_results['xgboost'] is not None:
    save_metrics('xgboost', ensemble_results['xgboost'], {
        'n_estimators': 150,
        'max_depth': 6,
        'learning_rate': 0.1,
    })

summary = {}
for name, result in ensemble_results.items():
    if result is None:
        continue
    summary[name] = {
        'accuracy': result['accuracy'],
        'f1_macro': result['f1_macro'],
        'train_time_sec': result['train_time_sec'],
        'pred_time_sec': result['pred_time_sec'],
    }

with open(metrics_dir / 'ensemble_comparison.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

print('Ensemble summary:')
for name, values in summary.items():
    print(f"{name:14s} | acc={values['accuracy']:.4f} | f1={values['f1_macro']:.4f} | train={values['train_time_sec']:.2f}s")

print('\nSaved files:')
for file_name in ['bagging_metrics.json', 'random_forest_metrics.json', 'adaboost_metrics.json', 'xgboost_metrics.json', 'ensemble_comparison.json']:
    full_path = metrics_dir / file_name
    if full_path.exists():
        print(full_path)

Ensemble summary:
bagging        | acc=0.9487 | f1=0.9481 | train=35.01s
random_forest  | acc=0.9687 | f1=0.9684 | train=5.82s
adaboost       | acc=0.8324 | f1=0.8319 | train=118.39s
xgboost        | acc=0.9695 | f1=0.9693 | train=192.53s

Saved files:
/Users/nrjsingh1/Library/Mobile Documents/com~apple~CloudDocs/MS_courses/CSIT_598_MachineLear/Assignments/project/results/metrics/bagging_metrics.json
/Users/nrjsingh1/Library/Mobile Documents/com~apple~CloudDocs/MS_courses/CSIT_598_MachineLear/Assignments/project/results/metrics/random_forest_metrics.json
/Users/nrjsingh1/Library/Mobile Documents/com~apple~CloudDocs/MS_courses/CSIT_598_MachineLear/Assignments/project/results/metrics/adaboost_metrics.json
/Users/nrjsingh1/Library/Mobile Documents/com~apple~CloudDocs/MS_courses/CSIT_598_MachineLear/Assignments/project/results/metrics/xgboost_metrics.json
/Users/nrjsingh1/Library/Mobile Documents/com~apple~CloudDocs/MS_courses/CSIT_598_MachineLear/Assignments/project/results/metrics/ensemb